<a href="https://colab.research.google.com/github/DiegoB2002/BUS4-118S/blob/dev/PromptEngineeringExercise1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1

In [6]:
# Colab-ready Customer Support Prompt Chain (NO API KEY)
# - Clear, role-appropriate prompts with constraints
# - Interactive: you paste a customer message + answer follow-up questions
# - Uses a small rule-based "fake LLM" so it runs anywhere
#
# Tip: set DEBUG=True to print the exact prompts used at each step.

import json
import re
from typing import Dict, Any, List

DEBUG = False  # <- set True to print prompts


# -----------------------------
# Helpers
# -----------------------------
def jprint(x):
    print(json.dumps(x, indent=2, ensure_ascii=False))

def norm(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip().lower())

def contains_any(text: str, keywords: List[str]) -> bool:
    t = norm(text)
    return any(k in t for k in keywords)

def safe_input(prompt: str) -> str:
    try:
        return input(prompt).strip()
    except EOFError:
        # In rare notebook contexts, input can fail; return empty gracefully
        return ""


# -----------------------------
# Prompt builders (clear roles + constraints)
# -----------------------------
def build_classification_prompt(customer_message: str) -> str:
    return f"""SYSTEM ROLE: You are a Tier-1 Customer Support AI for a SaaS company.

STEP 1: CLASSIFICATION
Goal: Classify the customer issue.

You MUST:
- Choose one primary category from: billing, login, bug, account, shipping, security, other
- Set severity: low, medium, high
- Add risk_flags from: security, fraud, privacy, legal (empty list if none)
- Provide a short reason (1-2 sentences)

Constraints:
- Do NOT propose solutions.
- Do NOT ask follow-up questions.
- Do NOT assume facts not in the message.
- Output MUST be valid JSON with keys:
  category, severity, confidence (0..1), risk_flags (list), reason (string)

CUSTOMER MESSAGE:
\"\"\"{customer_message}\"\"\"
"""

def build_missing_info_prompt(customer_message: str, classification: Dict[str, Any]) -> str:
    return f"""SYSTEM ROLE: You are a Tier-1 Customer Support AI.

STEP 2: MISSING INFORMATION
Goal: Ask only the minimum questions needed to resolve the issue.

You MUST:
- Ask <= 4 questions total
- Tailor questions to category: {classification["category"]}
- Ask for identifiers needed to look up the case (e.g., email, order/invoice id) when relevant

Constraints:
- Tone: polite, concise, professional
- Avoid sensitive data: passwords, full card numbers, OTP/2FA codes, SSNs
- Do NOT propose solutions yet
- Output MUST be valid JSON with keys:
  questions (list of strings), reason (string)

CUSTOMER MESSAGE:
\"\"\"{customer_message}\"\"\"

CLASSIFICATION:
{json.dumps(classification, indent=2, ensure_ascii=False)}
"""

def build_solution_prompt(customer_message: str,
                          classification: Dict[str, Any],
                          collected_info: Dict[str, str]) -> str:
    return f"""SYSTEM ROLE: You are a Tier-1 Customer Support AI.

STEP 3: SOLUTION PROPOSAL
Goal: Draft a customer-facing reply + internal action checklist.

You MUST:
- Use the classification + collected info
- Provide:
  1) customer_reply: <= 150 words
  2) internal_actions: list of concrete steps
  3) needs_customer_reply: true/false
  4) resolution_confidence: number 0..1

Constraints:
- Tone: calm, empathetic, professional
- Do NOT blame the customer
- Do NOT request sensitive data (passwords, full card numbers, OTP/2FA codes)
- If info is missing, ask ONLY for what’s missing (briefly)

Output MUST be valid JSON with keys:
customer_reply, internal_actions, needs_customer_reply, resolution_confidence

CUSTOMER MESSAGE:
\"\"\"{customer_message}\"\"\"

CLASSIFICATION:
{json.dumps(classification, indent=2, ensure_ascii=False)}

COLLECTED_INFO:
{json.dumps(collected_info, indent=2, ensure_ascii=False)}
"""

def build_escalation_prompt(classification: Dict[str, Any], solution: Dict[str, Any]) -> str:
    return f"""SYSTEM ROLE: You are a Support Operations AI.

STEP 4: ESCALATION CHECK
Goal: Decide whether to escalate and where.

Escalate IF ANY:
- severity == "high"
- risk_flags includes security/privacy/legal/fraud
- resolution_confidence < 0.5

Otherwise:
- keep Tier-1 (no escalation)

Constraints:
- Output MUST be valid JSON with keys:
  escalate (true/false), priority (P1/P2/P3), queue (string), reason (string)

CLASSIFICATION:
{json.dumps(classification, indent=2, ensure_ascii=False)}

SOLUTION:
{json.dumps(solution, indent=2, ensure_ascii=False)}
"""


# -----------------------------
# Fake "LLM" (rule-based) so it runs without API calls
# Swap this function with a real model call later if you want.
# -----------------------------
def fake_llm_step1_classify(customer_message: str) -> Dict[str, Any]:
    msg = norm(customer_message)
    risk_flags = []

    if contains_any(msg, ["hacked", "unauthorized", "phishing", "suspicious", "fraud"]):
        risk_flags.append("security")
    if contains_any(msg, ["gdpr", "ccpa", "delete my data", "privacy"]):
        risk_flags.append("privacy")
    if contains_any(msg, ["chargeback", "lawsuit", "attorney", "legal"]):
        risk_flags.append("legal")

    # category
    if contains_any(msg, ["refund", "charged", "billing", "invoice", "payment", "card", "subscription", "plan"]):
        category = "billing"
    elif contains_any(msg, ["can't log in", "cannot log in", "login", "sign in", "password", "2fa", "otp", "verification"]):
        category = "login"
    elif contains_any(msg, ["bug", "error", "crash", "broken", "not working", "500"]):
        category = "bug"
    elif contains_any(msg, ["cancel", "delete account", "change email", "account"]):
        category = "account"
    elif contains_any(msg, ["shipping", "delivery", "tracking", "package", "order"]):
        category = "shipping"
    elif "security" in risk_flags:
        category = "security"
    else:
        category = "other"

    # severity
    severity = "low"
    if "security" in risk_flags:
        severity = "high"
    elif contains_any(msg, ["urgent", "asap", "immediately", "blocked", "down", "can't access"]):
        severity = "medium"
    if contains_any(msg, ["charged twice", "lost money", "data loss", "cannot access account"]):
        severity = "high"

    # confidence (simple heuristic)
    confidence = 0.55
    if category != "other":
        confidence += 0.25
    if risk_flags:
        confidence += 0.1
    confidence = max(0.0, min(1.0, confidence))

    reason = f"Detected signals consistent with {category}."
    if risk_flags:
        reason += f" Risk flags: {', '.join(risk_flags)}."

    return {
        "category": category,
        "severity": severity,
        "confidence": round(confidence, 2),
        "risk_flags": risk_flags,
        "reason": reason
    }

def fake_llm_step2_missing_info(classification: Dict[str, Any]) -> Dict[str, Any]:
    cat = classification["category"]
    risk = set(classification["risk_flags"])

    # For security: minimal + safe questions
    if "security" in risk or cat == "security":
        return {
            "questions": [
                "Do you still have access to the account? (yes/no)",
                "When did you first notice the suspicious activity?",
                "What changes did you see (password/email/billing/etc.)?",
                "Have you reset your password and enabled 2FA? (yes/no)"
            ],
            "reason": "Security-related issue: collect only what’s needed to secure the account quickly."
        }

    # Category-specific (<=4)
    if cat == "billing":
        return {
            "questions": [
                "What email is on the account?",
                "Do you have an invoice/order ID (or last 6+ digits)?",
                "What was the date and amount of the charge/refund?",
                "Is this an unexpected charge, duplicate charge, or missing refund?"
            ],
            "reason": "Billing investigations require matching the account and transaction details."
        }
    if cat == "login":
        return {
            "questions": [
                "What email is on the account?",
                "Are you signing in on web, iOS, Android, or desktop app?",
                "Have you tried a password reset? If yes, what happened?",
                "What exact error message do you see (copy/paste)?"
            ],
            "reason": "Login issues are resolved faster with platform + error message + reset status."
        }
    if cat == "bug":
        return {
            "questions": [
                "What steps reproduce the issue? (1–2–3)",
                "What did you expect vs what actually happened?",
                "What device/OS/browser/app version are you using?",
                "Can you share the exact error text or a screenshot (no sensitive info)?"
            ],
            "reason": "Bug triage needs reproducible steps and environment details."
        }
    if cat == "shipping":
        return {
            "questions": [
                "What’s your order ID?",
                "What country is the shipping address in?",
                "Do you have a tracking number or carrier name?",
                "What’s the latest tracking status/date you see?"
            ],
            "reason": "Shipping support needs order + tracking details to locate the package status."
        }

    # other
    return {
        "questions": [
            "What email is on the account?",
            "What are you trying to do, and what happened instead?",
            "When did this start happening?",
        ],
        "reason": "Need a bit more detail to route and resolve appropriately."
    }

def fake_llm_step3_solution(classification: Dict[str, Any], info: Dict[str, str]) -> Dict[str, Any]:
    cat = classification["category"]
    risk = set(classification["risk_flags"])

    # Helper fields
    email = info.get("What email is on the account?", "").strip()
    order = info.get("Do you have an invoice/order ID (or last 6+ digits)?", "").strip() or info.get("What’s your order ID?", "").strip()
    date_amt = info.get("What was the date and amount of the charge/refund?", "").strip()
    issue_type = info.get("Is this an unexpected charge, duplicate charge, or missing refund?", "").strip()

    if "security" in risk or cat == "security":
        customer_reply = (
            "Thanks for reporting this — I’m going to help you secure the account. "
            "Please reset your password (use a unique one) and enable 2FA if available. "
            "Reply with: (1) whether you still have access, (2) when you noticed this, "
            "and (3) what changed (password/email/billing). We’ll then review recent activity and lock down access."
        )
        return {
            "customer_reply": customer_reply,
            "internal_actions": [
                "Tag ticket: Security",
                "Guide password reset + enable 2FA (never request OTP codes)",
                "Review account audit logs and active sessions; revoke sessions if supported",
                "Check for unauthorized billing changes; advise contacting bank if charges exist"
            ],
            "needs_customer_reply": True,
            "resolution_confidence": 0.55
        }

    if cat == "billing":
        needs = []
        if not email: needs.append("account email")
        if not order: needs.append("invoice/order id")
        if not date_amt: needs.append("charge date/amount")

        if needs:
            customer_reply = (
                "I can help with this billing issue. To locate the transaction, please reply with "
                f"{', '.join(needs)}. Once I have that, I’ll confirm what happened and take the next step (e.g., refund or correction) per policy."
            )
            needs_reply = True
            conf = 0.5
        else:
            customer_reply = (
                f"Thanks — I’ve got your details (email: {email}, order: {order}, charge: {date_amt}). "
                f"I’ll verify whether this is a {issue_type or 'billing issue'} and confirm the transaction status. "
                "If it’s a duplicate or incorrect charge, we’ll initiate a refund and share the timeline."
            )
            needs_reply = False
            conf = 0.8

        return {
            "customer_reply": customer_reply,
            "internal_actions": [
                f"Lookup customer by email: {email or '[missing]'}",
                f"Find transaction by order/invoice: {order or '[missing]'}",
                f"Verify charge/refund details: {date_amt or '[missing]'}",
                "Apply refund/correction per billing policy; document outcome"
            ],
            "needs_customer_reply": needs_reply,
            "resolution_confidence": conf
        }

    # default generic
    customer_reply = (
        "Thanks — I can help. Based on what you shared, I may need one or two more details to move forward. "
        "Please answer the questions above and I’ll propose the next best step."
    )
    return {
        "customer_reply": customer_reply,
        "internal_actions": ["Collect missing info", "Route to correct playbook", "Proceed with resolution"],
        "needs_customer_reply": True,
        "resolution_confidence": 0.55
    }

def fake_llm_step4_escalation(classification: Dict[str, Any], solution: Dict[str, Any]) -> Dict[str, Any]:
    sev = classification["severity"]
    risk = classification["risk_flags"]
    conf = float(solution.get("resolution_confidence", 0.0))

    if sev == "high" or any(r in ["security", "privacy", "legal", "fraud"] for r in risk):
        return {
            "escalate": True,
            "priority": "P1" if sev == "high" or "security" in risk else "P2",
            "queue": "Security Response" if "security" in risk else ("Privacy & Compliance" if "privacy" in risk else "L2 Support"),
            "reason": "Escalate due to high severity and/or risk flags."
        }
    if conf < 0.5:
        return {
            "escalate": True,
            "priority": "P2",
            "queue": "L2 Support",
            "reason": "Low resolution confidence at Tier-1."
        }
    return {
        "escalate": False,
        "priority": "P3",
        "queue": "Tier-1",
        "reason": "No risk flags and sufficient confidence to continue at Tier-1."
    }


# -----------------------------
# Chain runner (interactive)
# -----------------------------
def run_chain():
    customer_message = safe_input("Paste the customer's message:\n> ")
    if not customer_message:
        print("No message provided; stopping.")
        return

    # STEP 1
    p1 = build_classification_prompt(customer_message)
    if DEBUG:
        print("\n--- STEP 1 PROMPT ---\n", p1)
    classification = fake_llm_step1_classify(customer_message)

    # STEP 2
    p2 = build_missing_info_prompt(customer_message, classification)
    if DEBUG:
        print("\n--- STEP 2 PROMPT ---\n", p2)
    missing_info = fake_llm_step2_missing_info(classification)

    # Ask questions interactively (this is the "gather missing info" part)
    collected: Dict[str, str] = {}
    print("\n" + "=" * 70)
    print("STEP 2 — QUESTIONS TO ASK CUSTOMER")
    print("=" * 70)
    print(missing_info["reason"])
    for q in missing_info["questions"]:
        collected[q] = safe_input(f"\n{q}\n> ")

    # STEP 3
    p3 = build_solution_prompt(customer_message, classification, collected)
    if DEBUG:
        print("\n--- STEP 3 PROMPT ---\n", p3)
    solution = fake_llm_step3_solution(classification, collected)

    # STEP 4
    p4 = build_escalation_prompt(classification, solution)
    if DEBUG:
        print("\n--- STEP 4 PROMPT ---\n", p4)
    escalation = fake_llm_step4_escalation(classification, solution)

    # Print outputs
    print("\n" + "=" * 70)
    print("STEP 1 — CLASSIFICATION (OUTPUT)")
    print("=" * 70)
    jprint(classification)

    print("\n" + "=" * 70)
    print("STEP 2 — COLLECTED INFO (OUTPUT)")
    print("=" * 70)
    jprint(collected)

    print("\n" + "=" * 70)
    print("STEP 3 — CUSTOMER REPLY + INTERNAL ACTIONS (OUTPUT)")
    print("=" * 70)
    jprint(solution)

    print("\n" + "=" * 70)
    print("STEP 4 — ESCALATION DECISION (OUTPUT)")
    print("=" * 70)
    jprint(escalation)


# Run it
run_chain()

Paste the customer's message:
> billing

STEP 2 — QUESTIONS TO ASK CUSTOMER
Billing investigations require matching the account and transaction details.

What email is on the account?
> diego.easdas@gmail.com

Do you have an invoice/order ID (or last 6+ digits)?
> 123799

What was the date and amount of the charge/refund?
> 12/12/12 $109

Is this an unexpected charge, duplicate charge, or missing refund?
> missing refund

STEP 1 — CLASSIFICATION (OUTPUT)
{
  "category": "billing",
  "severity": "low",
  "confidence": 0.8,
  "risk_flags": [],
  "reason": "Detected signals consistent with billing."
}

STEP 2 — COLLECTED INFO (OUTPUT)
{
  "What email is on the account?": "diego.easdas@gmail.com",
  "Do you have an invoice/order ID (or last 6+ digits)?": "123799",
  "What was the date and amount of the charge/refund?": "12/12/12 $109",
  "Is this an unexpected charge, duplicate charge, or missing refund?": "missing refund"
}

STEP 3 — CUSTOMER REPLY + INTERNAL ACTIONS (OUTPUT)
{
  "custome